# master_data.csv 생성
## 지정학적 위기 시 비트코인은 디지털 금인가?
### 캡스톤디자인 | 팀명: 분석많이된다

---

## 출력 파일

```
master_data.csv
```

## 흐름

```
Step 0. 라이브러리 설치 및 로드
Step 1. market_returns.csv 수집
Step 2. gpr_combined.csv 로드
Step 3. VIX 수집
Step 4. Fear & Greed 수집
Step 5. 병합
Step 6. 결측치 처리
Step 7. 검증 및 저장
```

---
## Step 0. 라이브러리 설치 및 로드

In [1]:
# !pip install yfinance pandas numpy requests --quiet

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
import os
warnings.filterwarnings('ignore')

EVENT_DATES = {
    'hormuz_crisis'          : '2019-06-13',
    'soleimani_assassination': '2020-01-03',
    'russia_ukraine_war': '2022-02-24',
    'israel_hamas_war'           : '2023-10-07',
    'israel_iran'            : '2024-04-01',
    'us_israel_iran'         : '2026-02-28',
}

GLOBAL_START = '2018-12-31'
GLOBAL_END   = '2026-05-01'

print('✅ 라이브러리 로드 완료')
print(f'   수집 기간: {GLOBAL_START} ~ {GLOBAL_END}')

✅ 라이브러리 로드 완료
   수집 기간: 2018-12-31 ~ 2026-05-01


---
## Step 1. market_returns.csv 수집

In [ ]:
TICKERS = {
    'BTC'   : 'BTC-USD',
    'Gold'  : 'GC=F',
    'TLT'   : 'TLT',
    'DXY'   : 'DX-Y.NYB',
    'SP500' : '^GSPC',
    'NASDAQ': '^IXIC',
}

print('▶ yfinance 가격 데이터 수집 중...')
raw = yf.download(
    list(TICKERS.values()),
    start=GLOBAL_START,
    end=GLOBAL_END,
    auto_adjust=False,
    progress=False
)['Close']
raw = raw[list(TICKERS.values())]
raw.columns = list(TICKERS.keys())

is_trading_day = raw['SP500'].notna()
trading_days = raw[is_trading_day].index
non_trading_days = raw[~is_trading_day].index

btc_all_returns = np.log(raw['BTC'] / raw['BTC'].shift(1))


processed_btc_returns = []
all_dates = raw.index.tolist()

i = 0
while i < len(all_dates):
    current_date = all_dates[i]
    
    if is_trading_day.loc[current_date]:
        combined_return = btc_all_returns.loc[current_date]
        if pd.isna(combined_return): combined_return = 0
        
        processed_btc_returns.append(combined_return)
        i += 1
    else:
        j = i
        next_trading_idx = i
        while next_trading_idx < len(all_dates) and not is_trading_day.loc[all_dates[next_trading_idx]]:
            next_trading_idx += 1
        
        if next_trading_idx < len(all_dates):
            current_ret = btc_all_returns.loc[all_dates[i]]
            if not pd.isna(current_ret):
                btc_all_returns.loc[all_dates[next_trading_idx]] += current_ret
        
        i += 1

prices = raw.loc[trading_days].ffill()
returns = np.log(prices / prices.shift(1))

if len(processed_btc_returns) > len(returns):
    returns['BTC'] = processed_btc_returns[1:]
else:
    returns['BTC'] = processed_btc_returns

returns = returns.dropna()
returns.index.name = 'date'

returns.to_csv('./processed_data/market_returns.csv', encoding='utf-8-sig')

print(f'✅ market_returns.csv 업데이트 완료')
print(f'   기간: {returns.index.min().date()} ~ {returns.index.max().date()}')
print(f'   행 수: {len(returns)}일')
print(f'   결측치: {returns.isnull().sum().sum()}건')
print(f'\n기술통계:')
print(returns.describe().round(4))

▶ yfinance 가격 데이터 수집 중...
✅ market_returns.csv 업데이트 완료
   기간: 2019-01-02 ~ 2026-04-30
   행 수: 1842일
   결측치: 0건

기술통계:
             BTC       Gold        TLT        DXY      SP500     NASDAQ
count  1842.0000  1842.0000  1842.0000  1842.0000  1842.0000  1842.0000
mean      0.0016     0.0007    -0.0002     0.0000     0.0006     0.0007
std       0.0396     0.0113     0.0102     0.0043     0.0125     0.0151
min      -0.4647    -0.1207    -0.0690    -0.0214    -0.1277    -0.1315
25%      -0.0160    -0.0044    -0.0064    -0.0025    -0.0044    -0.0060
50%       0.0009     0.0010     0.0000     0.0000     0.0010     0.0014
75%       0.0196     0.0065     0.0058     0.0025     0.0067     0.0087
max       0.2030     0.0591     0.0725     0.0164     0.0909     0.1148


---
## Step 2. gpr_combined.csv 로드



In [3]:
GPR_FILE = './processed_data/final/gpr_combined.csv'

if not os.path.exists(GPR_FILE):
    raise FileNotFoundError(
        f'{GPR_FILE} 없음.\nGPR_custom_analysis.ipynb를 먼저 실행하세요.'
    )

gpr = pd.read_csv(GPR_FILE)
gpr['date'] = pd.to_datetime(gpr['date'])

for col in ['F3_z','F3_raw','GPR','GPR_zscore','N','mean_tone']:
    if col in gpr.columns:
        gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

gpr['GPR_custom'] = gpr['F3_z']

print('✅ gpr_combined.csv 로드 완료')
print(f'   행 수: {len(gpr)}')
print(f'   기간: {gpr["date"].min().date()} ~ {gpr["date"].max().date()}')
print(f'\n이벤트별 일수:')
print(gpr.groupby('event_name')['date'].count().to_string())

✅ gpr_combined.csv 로드 완료
   행 수: 2605
   기간: 2019-01-01 ~ 2026-04-30

이벤트별 일수:
event_name
hormuz_crisis              266
israel_hamas_war           384
israel_iran                438
russia_ukraine_war         686
soleimani_assassination    438
us_israel_iran             393


---
## Step 3. VIX 수집


In [4]:
GLOBAL_START = '2019-01-01'
print('▶ VIX 수집 중...')
vix_raw = yf.download(
    '^VIX',
    start=GLOBAL_START,
    end=GLOBAL_END,
    
    auto_adjust=False,
    progress=False
)['Close']

if hasattr(vix_raw, 'squeeze'):
    vix_raw = vix_raw.squeeze()

vix = pd.DataFrame({'VIX': vix_raw})
vix.index = pd.to_datetime(vix.index)
vix.index.name = 'date'
vix['VIX'] = pd.to_numeric(vix['VIX'], errors='coerce')
vix = vix.dropna().reset_index()

print(f'✅ VIX 수집 완료')
print(f'   기간: {vix["date"].min().date()} ~ {vix["date"].max().date()}')
print(f'   행 수: {len(vix)}')
print(vix['VIX'].describe().round(2))

▶ VIX 수집 중...
✅ VIX 수집 완료
   기간: 2019-01-02 ~ 2026-04-30
   행 수: 1842
count    1842.00
mean       20.21
std         7.52
min        11.54
25%        15.35
50%        18.27
75%        23.03
max        82.69
Name: VIX, dtype: float64


---
## Step 4. Fear & Greed Index 수집


In [5]:
def fetch_fear_greed(limit=3000):
    url = f'https://api.alternative.me/fng/?limit={limit}&format=json'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()['data']
        fg = pd.DataFrame(data)
        fg['date']       = pd.to_datetime(fg['timestamp'].astype(int), unit='s').dt.normalize()
        fg['fear_greed'] = fg['value'].astype(int)
        fg['fg_label']   = fg['value_classification']
        fg = (fg[['date','fear_greed','fg_label']]
              .sort_values('date')
              .drop_duplicates(subset=['date'])
              .reset_index(drop=True))
        return fg
    except Exception as e:
        print(f'⚠️ API 오류: {e}')
        return pd.DataFrame(columns=['date','fear_greed','fg_label'])

print('▶ Fear & Greed 수집 중...')
fg = fetch_fear_greed(limit=3000)

if len(fg) > 0:
    fg['date'] = pd.to_datetime(fg['date'])
    fg_range = fg[fg['date'] >= pd.Timestamp(GLOBAL_START)]
    print(f'✅ Fear & Greed 수집 완료')
    print(f'   전체: {fg["date"].min().date()} ~ {fg["date"].max().date()}')
    print(f'   분석 기간 내: {len(fg_range)}일')
    print(fg_range['fear_greed'].describe().round(1))
else:
    print('⚠️ 수집 실패 → NaN으로 처리됩니다')

▶ Fear & Greed 수집 중...
✅ Fear & Greed 수집 완료
   전체: 2018-03-10 ~ 2026-05-30
   분석 기간 내: 2706일
count    2706.0
mean       47.8
std        22.1
min         5.0
25%        28.0
50%        48.0
75%        68.0
max        95.0
Name: fear_greed, dtype: float64


---
## Step 5. 병합


In [6]:
returns_df = returns.copy().reset_index()
if 'Date' in returns_df.columns:
    returns_df = returns_df.rename(columns={'Date': 'date'})
returns_df['date'] = pd.to_datetime(returns_df['date'])

vix_df = vix.copy()
vix_df['date'] = pd.to_datetime(vix_df['date'])

fg_df = fg.copy() if len(fg) > 0 else pd.DataFrame(columns=['date','fear_greed','fg_label'])
fg_df['date'] = pd.to_datetime(fg_df['date'])

gpr_cols = ['date','event_name','event_date',
            'GPR_custom','F3_raw','GPR','GPR_zscore','N','mean_tone']
gpr_use = gpr[[c for c in gpr_cols if c in gpr.columns]].copy()

master = gpr_use.copy()

master = master.merge(returns_df, on='date', how='left')
print(f'returns 병합: {len(master)}행  수익률 매칭 {master["BTC"].notna().sum()}행')

master = master.merge(vix_df[['date','VIX']], on='date', how='left')
print(f'VIX 병합   : {len(master)}행  VIX 매칭 {master["VIX"].notna().sum()}행')

if len(fg_df) > 0:
    master = master.merge(fg_df[['date','fear_greed','fg_label']], on='date', how='left')
    print(f'F&G 병합   : {len(master)}행  F&G 매칭 {master["fear_greed"].notna().sum()}행')
else:
    master['fear_greed'] = np.nan
    master['fg_label']   = np.nan

print(f'\n병합 직후 컬럼: {master.columns.tolist()}')

returns 병합: 2605행  수익률 매칭 1800행
VIX 병합   : 2605행  VIX 매칭 1800행
F&G 병합   : 2605행  F&G 매칭 2604행

병합 직후 컬럼: ['date', 'event_name', 'event_date', 'GPR_custom', 'F3_raw', 'GPR', 'GPR_zscore', 'N', 'mean_tone', 'BTC', 'Gold', 'TLT', 'DXY', 'SP500', 'NASDAQ', 'VIX', 'fear_greed', 'fg_label']


---
## Step 6. 결측치 처리


In [7]:
print('▶ 결측치 처리 전 현황:')
key_cols = ['BTC','Gold','SP500','GPR_custom','VIX','fear_greed']
key_cols = [c for c in key_cols if c in master.columns]
print(master[key_cols].isnull().sum().to_string())
print(f'전체 행: {len(master)}')

▶ 결측치 처리 전 현황:
BTC           805
Gold          805
SP500         805
GPR_custom      0
VIX           805
fear_greed      1
전체 행: 2605


In [8]:
before = len(master)
master = master.dropna(subset=['BTC', 'SP500']).copy()
after  = len(master)
print(f'① BTC·SP500 결측 제거: {before}행 → {after}행 ({before-after}행 제거)')

① BTC·SP500 결측 제거: 2605행 → 1800행 (805행 제거)


In [9]:
master = master.sort_values(['event_name', 'date']).reset_index(drop=True)
vix_before = master['VIX'].isna().sum()
master['VIX'] = master.groupby('event_name')['VIX'].ffill().bfill()
vix_after = master['VIX'].isna().sum()
print(f'② VIX ffill: {vix_before}건 → {vix_after}건 결측')

② VIX ffill: 0건 → 0건 결측


In [10]:
fg_before = master['fear_greed'].isna().sum()
master['fear_greed'] = master.groupby('event_name')['fear_greed'].ffill().bfill()
fg_after = master['fear_greed'].isna().sum()
print(f'③ Fear&Greed ffill: {fg_before}건 → {fg_after}건 결측')

master['fear_greed_lag1'] = master.groupby('event_name')['fear_greed'].shift(1)
master['fear_greed_lag1'] = master.groupby('event_name')['fear_greed_lag1'].ffill()
print(f'   fear_greed_lag1 결측: {master["fear_greed_lag1"].isna().sum()}건')

③ Fear&Greed ffill: 0건 → 0건 결측
   fear_greed_lag1 결측: 6건


In [11]:
print('\n▶ 결측치 처리 후 현황:')
all_cols = ['BTC','Gold','TLT','DXY','SP500','NASDAQ',
            'GPR_custom','GPR','VIX','fear_greed','fear_greed_lag1']
all_cols = [c for c in all_cols if c in master.columns]
na_summary = master[all_cols].isnull().sum()
for col, n in na_summary.items():
    flag = '✅' if n == 0 else f'⚠️  {n}건'
    print(f'  {col:<20}: {flag}')

print(f'\n최종 행 수: {len(master)}')
print(f'\n이벤트별 행 수 (거래일 기준):')
print(master.groupby('event_name')['date'].count().to_string())


▶ 결측치 처리 후 현황:
  BTC                 : ✅
  Gold                : ✅
  TLT                 : ✅
  DXY                 : ✅
  SP500               : ✅
  NASDAQ              : ✅
  GPR_custom          : ✅
  GPR                 : ✅
  VIX                 : ✅
  fear_greed          : ✅
  fear_greed_lag1     : ⚠️  6건

최종 행 수: 1800

이벤트별 행 수 (거래일 기준):
event_name
hormuz_crisis              183
israel_hamas_war           262
israel_iran                299
russia_ukraine_war         475
soleimani_assassination    309
us_israel_iran             272


---
## Step 7. 검증 및 저장

In [12]:
num_cols = ['BTC','Gold','TLT','DXY','SP500','NASDAQ',
            'GPR_custom','GPR','VIX','fear_greed']
num_cols = [c for c in num_cols if c in master.columns]

for col in num_cols:
    master[col] = pd.to_numeric(master[col], errors='coerce')

print('▶ 기술통계:')
display(master[num_cols].describe().round(4))

▶ 기술통계:


,BTC,Gold,TLT,DXY,SP500,NASDAQ,GPR_custom,GPR,VIX,fear_greed
count,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000,1800.0000
mean,0.0017,0.0007,-0.0002,0.0000,0.0005,0.0006,0.0784,120.4075,20.1223,47.7794
std,0.0399,0.0114,0.0103,0.0043,0.0125,0.0152,1.0106,48.4260,7.4704,22.0625
min,-0.4647,-0.1207,-0.0690,-0.0214,-0.1277,-0.1315,-2.2631,58.4200,11.5400,5.0000
25%,-0.0161,-0.0044,-0.0064,-0.0025,-0.0045,-0.0060,-0.5743,86.5700,15.2975,28.0000
50%,0.0010,0.0010,0.0000,0.0001,0.0009,0.0013,-0.2796,110.5300,18.2150,49.0000
75%,0.0199,0.0065,0.0058,0.0025,0.0065,0.0087,0.3857,138.6700,22.8750,68.0000
max,0.2030,0.0591,0.0725,0.0164,0.0909,0.1148,7.7082,326.9900,82.6900,95.0000


In [19]:
print('▶ 이벤트별 샘플 (각 2행):\n')
show_cols = ['date','event_name','BTC','SP500',
             'GPR_custom','VIX','fear_greed']
show_cols = [c for c in show_cols if c in master.columns]

for ev in EVENT_DATES:
    sub = master[master['event_name'] == ev]
    if len(sub) == 0:
        print(f'[{ev}] ⚠️ 데이터 없음')
        continue
    print(f'[{ev}]  {len(sub)}거래일')
    print(sub[show_cols].head(2).to_string(index=False))
    print()

▶ 이벤트별 샘플 (각 2행):

[hormuz_crisis]  183거래일
      date    event_name       BTC     SP500  GPR_custom       VIX  fear_greed
2019-01-02 hormuz_crisis  0.052238  0.001268   -0.745422 23.219999        30.0
2019-01-03 hormuz_crisis -0.027422 -0.025068    0.669603 25.450001        33.0

[soleimani_assassination]  309거래일
      date              event_name       BTC     SP500  GPR_custom       VIX  fear_greed
2019-09-24 soleimani_assassination -0.120994 -0.008452   -0.558029 17.049999        39.0
2019-09-25 soleimani_assassination -0.015616  0.006140   -0.605363 15.960000        15.0

[russia_ukraine_war]  475거래일
      date         event_name       BTC     SP500  GPR_custom   VIX  fear_greed
2021-01-29 russia_ukraine_war  0.025090 -0.019500   -0.551123 33.09        77.0
2021-02-01 russia_ukraine_war -0.022968  0.015924   -0.251112 30.24        77.0

[israel_hamas_war]  262거래일
      date       event_name       BTC     SP500  GPR_custom       VIX  fear_greed
2022-12-16 israel_hamas_war -0.042190 

In [ ]:
master.to_csv('./processed_data/final/master_data.csv', index=False, encoding='utf-8-sig')

print('=' * 55)
print('✅ master_data.csv 저장 완료')
print('=' * 55)
print(f'   행 수  : {len(master)}')
print(f'   컬럼 수: {len(master.columns)}')
print(f'\n컬럼 목록:')
for col in master.columns:
    na  = master[col].isnull().sum()
    flag = '✅' if na == 0 else f'⚠️  {na}건 결측'
    print(f'  {col:<25} {flag}')

✅ master_data.csv 저장 완료
   행 수  : 1800
   컬럼 수: 19

컬럼 목록:
  date                      ✅
  event_name                ✅
  event_date                ✅
  GPR_custom                ✅
  F3_raw                    ✅
  GPR                       ✅
  GPR_zscore                ✅
  N                         ✅
  mean_tone                 ✅
  BTC                       ✅
  Gold                      ✅
  TLT                       ✅
  DXY                       ✅
  SP500                     ✅
  NASDAQ                    ✅
  VIX                       ✅
  fear_greed                ✅
  fg_label                  ✅
  fear_greed_lag1           ⚠️  6건 결측
